# Backpropagation — Part 3

---

In Parts 1 and 2, we computed gradients and updated weights. In this part, we build the **intuition** for *why* gradient descent works — by understanding the loss function as a geometric surface, what slopes mean, and what convergence actually is.

---

## 1. The Loss Function is a Surface

We are working with the same dataset and architecture from Parts 1 and 2:

- **Inputs:** CGPA ($x_1$), Profile Score ($x_2$)
- **Target:** LPA ($y$)
- **9 trainable parameters:** $w^{1}_{11},\ w^{1}_{21},\ w^{1}_{12},\ w^{1}_{22},\ w^{2}_{11},\ w^{2}_{21},\ b_{11},\ b_{12},\ b_{21}$

![Architecture](./images/neural_network_backprop.svg)

The forward pass equations are:

$$o_{11} = w^{1}_{11} x_1 + w^{1}_{21} x_2 + b_{11}$$

$$o_{12} = w^{1}_{12} x_1 + w^{1}_{22} x_2 + b_{12}$$

$$\hat{y} = w^{2}_{11} \cdot o_{11} + w^{2}_{21} \cdot o_{12} + b_{21}$$

Now substitute $o_{11}$ and $o_{12}$ into $\hat{y}$:

$$\hat{y} = w^{2}_{11}(w^{1}_{11} x_1 + w^{1}_{21} x_2 + b_{11}) + w^{2}_{21}(w^{1}_{12} x_1 + w^{1}_{22} x_2 + b_{12}) + b_{21}$$

And the loss is:

$$L = (y - \hat{y})^2$$

Expanding fully, $\hat{y}$ is a function of all 9 parameters. So the loss $L$ is also a function of all 9:

$$L = f\!\left(w^{1}_{11},\ w^{1}_{21},\ w^{1}_{12},\ w^{1}_{22},\ w^{2}_{11},\ w^{2}_{21},\ b_{11},\ b_{12},\ b_{21}\right)$$

> **Key insight:** The loss function is not a simple 2D curve — it is a **9-dimensional surface**. Every combination of weights and biases gives a different loss value. Our goal is to find the combination that makes this loss as small as possible.

---

## 2. Slopes in 9 Dimensions

Since $L$ depends on 9 parameters, it has **9 partial derivatives** (slopes) — one for each parameter. This is exactly what we computed in Parts 1 and 2.

Each partial derivative answers: *"If I change this one parameter slightly, how much does the loss change?"*

| Gradient | Meaning |
|----------|---------|
| $\frac{\partial L}{\partial w^{2}_{11}}$ | How much does $L$ change if we nudge $w^{2}_{11}$? |
| $\frac{\partial L}{\partial b_{21}}$ | How much does $L$ change if we nudge $b_{21}$? |
| $\vdots$ | $\vdots$ |

The collection of all 9 gradients is called the **gradient vector** $\nabla L$. It points in the direction of **steepest increase** of the loss. To minimize, we move in the **opposite direction**.

---

## 3. Simple Example — Loss as a Function of $b_{21}$ Only

To build intuition, let's imagine all other parameters are fixed and we only vary $b_{21}$.

With everything else held constant, $\hat{y}$ becomes:

$$\hat{y} = C + b_{21}$$

where $C$ is some fixed constant (all the weight terms). So the loss becomes:

$$L = (y - \hat{y})^2 = (y - C - b_{21})^2$$

Let $k = y - C$ (a constant). Then:

$$L = (k - b_{21})^2$$

This is a **parabola** in $b_{21}$ — a U-shaped curve with a single minimum at $b_{21} = k$.

The gradient is:

$$\frac{\partial L}{\partial b_{21}} = -2(k - b_{21}) = -2(y - \hat{y})$$

And the update rule is:

$$b_{21,\text{new}} = b_{21,\text{old}} - \eta \cdot \frac{\partial L}{\partial b_{21}}$$

Visually:

```
      L
      │        ●  ← starting point (high loss, steep slope)
      │      ╱   ╲
      │    ╱       ╲
      │  ╱           ╲
      │╱       ★       ╲   ← minimum (slope = 0)
      └──────────────────── b₂₁
                  ↑
             optimal b₂₁
```

- If we are on the **right** side of the minimum, the slope is positive → we subtract → $b_{21}$ decreases → we move left ✓  
- If we are on the **left** side, the slope is negative → we subtract a negative → $b_{21}$ increases → we move right ✓  
- The learning rate $\eta$ controls the **step size** at each update

The real 9D loss surface works by the same principle, just extended to all 9 parameters simultaneously.

---

## 4. What is Convergence?

**Convergence** is when the gradient (slope) approaches zero — meaning the loss has stopped decreasing and we have (ideally) reached a minimum.

$$\text{Converged} \iff \left\|\nabla L\right\| \approx 0 \quad \Longleftrightarrow \quad \frac{\partial L}{\partial w} \approx 0 \text{ for all weights}$$

### What convergence looks like in practice

```
Loss
  │
  │ ●
  │   ●
  │     ●
  │       ●●
  │          ●●●
  │              ●●●●●●●●●●  ← flattening out = convergence
  └────────────────────────────── Epoch
```

The loss curve flattens when all slopes are near zero. At that point, updates become tiny:

$$w_{\text{new}} = w_{\text{old}} - \eta \cdot \underbrace{\frac{\partial L}{\partial w}}_{\approx\ 0} \approx w_{\text{old}}$$

So the weights **stop changing** — the network has converged.

### Types of convergence

| Type | Description |
|------|-------------|
| **Global minimum** | The absolute lowest point on the loss surface. Guaranteed for convex functions (like pure MSE with linear networks). |
| **Local minimum** | A valley that is lower than its surroundings, but not the lowest overall. Common in deep networks with non-linear activations. |
| **Saddle point** | Slope is zero but it is not a minimum — flat in some directions, sloping in others. Can trap optimizers briefly. |

For our simple 2-layer linear network, the loss surface is **convex** (bowl-shaped), so gradient descent is guaranteed to find the global minimum.

---

## 5. Role of the Learning Rate $\eta$

The learning rate directly controls whether and how fast we converge.

```
Too large η:          Too small η:         Just right η:
                                          
Loss                  Loss                 Loss
│  ●                  │●                   │●
│    ●  ●             │ ●                  │  ●
│      ●  ●           │  ●                 │    ●●
│    ●  ●  ●          │   ●                │       ●●●●●●
│  ●  overshoots!     │    ●               │             ─── converges
│                     │     ... very slow  │
└─────────────        └─────────────       └─────────────
```

| Learning Rate | Effect |
|---------------|--------|
| Too large | Loss oscillates or diverges (overshoots the minimum) |
| Too small | Converges very slowly, wastes computation |
| Just right | Smooth, steady decrease toward the minimum |

A common strategy is to start with a moderate value (e.g. $\eta = 0.001$) and reduce it over time — this is called a **learning rate schedule**.

---

## 6. Full Picture — Putting It All Together

$$\underbrace{L = (y - \hat{y})^2}_{\text{9D surface}} \xrightarrow{\text{compute } \nabla L} \underbrace{\frac{\partial L}{\partial \theta_i}\ \text{for all 9 } \theta_i}_{\text{backpropagation}} \xrightarrow{\text{update}} \underbrace{\theta_i \leftarrow \theta_i - \eta \cdot \frac{\partial L}{\partial \theta_i}}_{\text{gradient descent}} \xrightarrow{\text{repeat}} \underbrace{\nabla L \approx 0}_{\text{convergence}}$$

Each training iteration:
1. **Forward pass** — compute $\hat{y}$ given current weights
2. **Loss** — measure how wrong $\hat{y}$ is
3. **Backward pass** — compute all 9 slopes via chain rule
4. **Update** — nudge each parameter downhill
5. **Repeat** — until slopes are near zero (converged)

---

## 7. Key Takeaways

- The loss function is a **high-dimensional surface** — one dimension per trainable parameter. With 9 parameters, it is a 9D surface.
- Each **partial derivative** tells us the slope of that surface along one parameter axis.
- **Gradient descent** moves all parameters simultaneously downhill by subtracting their scaled gradients.
- **Convergence** means the slopes have reached (approximately) zero — the loss can no longer decrease by nudging the weights.
- For our linear network, the surface is **convex** so there is one global minimum. For deeper non-linear networks, local minima and saddle points appear — but gradient descent still works well in practice.